# Chapter 9: Trajectory Generation
## Modern Robotics (Lynch & Park)

이 노트북은 Modern Robotics 교재 9장의 궤적 생성(Trajectory Generation) 내용을 다룹니다.

**주요 내용:**
1. Quintic Polynomial Trajectory (5차 다항식)
2. Trapezoidal Velocity Profile (사다리꼴 속도 프로파일)
3. S-Curve Velocity Profile
4. Minimum Jerk Trajectory
5. Linear Interpolation (선형 보간)
6. Multi-Waypoint Trajectory
7. B-Spline Curve

**Reference:** *Modern Robotics: Mechanics, Planning, and Control* (Lynch & Park), Chapter 9

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate

np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rc('xtick', labelsize=8)
plt.rc('ytick', labelsize=8)
%matplotlib inline

print("Setup complete.")

## 1. Quintic Polynomial Trajectory (5차 다항식 궤적)

MR 교재 9.2절: 시작/끝 위치, 속도, 가속도 조건을 만족하는 5차 다항식 궤적.

$$\theta(t) = a_0 + a_1 t + a_2 t^2 + a_3 t^3 + a_4 t^4 + a_5 t^5$$

6개의 경계 조건으로 6개의 계수를 결정합니다.

In [ ]:
def quintic_trajectory(start_pos, start_vel, start_acc,
                       end_pos, end_vel, end_acc,
                       duration, num_points):
    """5차 다항식 궤적 생성 (MR Ch9.2)"""
    n_joints = len(start_pos)
    t = np.linspace(0, duration, num_points)
    T = duration

    joint_coeffs = []
    for i in range(n_joints):
        A = np.array([
            [0,       0,       0,       0,     0, 1],
            [0,       0,       0,       0,     1, 0],
            [0,       0,       0,       2,     0, 0],
            [T**5,    T**4,    T**3,    T**2,  T, 1],
            [5*T**4,  4*T**3,  3*T**2,  2*T,   1, 0],
            [20*T**3, 12*T**2, 6*T,     2,     0, 0],
        ])
        b = np.array([start_pos[i], start_vel[i], start_acc[i],
                       end_pos[i],   end_vel[i],   end_acc[i]])
        x = np.linalg.solve(A, b)
        joint_coeffs.append(x)

    positions     = np.zeros((num_points, n_joints))
    velocities    = np.zeros((num_points, n_joints))
    accelerations = np.zeros((num_points, n_joints))
    jerks         = np.zeros((num_points, n_joints))

    for i in range(num_points):
        for j in range(n_joints):
            c = joint_coeffs[j]
            positions[i, j]     = np.polyval(c, t[i])
            velocities[i, j]    = np.polyval(np.polyder(c), t[i])
            accelerations[i, j] = np.polyval(np.polyder(c, 2), t[i])
            jerks[i, j]         = np.polyval(np.polyder(c, 3), t[i])

    return t, positions, velocities, accelerations, jerks

In [ ]:
# 예제: 6-DOF 로봇 조인트 궤적
q_start = np.array([-1.5705, -2.3119, 2.1442, 1.7392, 0.7854, -1.5711])
q_end   = np.array([-0.45,   -0.59,   0.60,  -0.01,  1.13,    0.0])

duration = 3.0
num_points = 500

t, pos, vel, acc, jerk = quintic_trajectory(
    q_start, np.zeros(6), np.zeros(6),
    q_end,   np.zeros(6), np.zeros(6),
    duration, num_points
)

fig, axes = plt.subplots(1, 4, figsize=(22, 4))
titles = ["Joint Positions", "Joint Velocities", "Joint Accelerations", "Joint Jerks"]
data = [pos, vel, acc, jerk]
ylabels = ["Position [rad]", "Velocity [rad/s]", r"Acceleration [rad/s$^2$]", r"Jerk [rad/s$^3$]"]

for ax, title, d, ylabel in zip(axes, titles, data, ylabels):
    for i in range(6):
        ax.plot(t, d[:, i], label=f"$q_{i+1}$")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc='best', fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Trapezoidal Velocity Profile (사다리꼴 속도 프로파일)

MR 교재 9.3절: 가속-등속-감속 3구간으로 구성된 속도 프로파일.

- 가속 구간: $0 \le t < t_a$
- 등속 구간: $t_a \le t < t_a + t_c$
- 감속 구간: $t_a + t_c \le t \le T$

In [ ]:
def trapezoidal_velocity_profile(d_total, v_max, a_max, n_points=1000):
    """사다리꼴 속도 프로파일 생성 (MR Ch9.3)"""
    t_ramp = v_max / a_max
    d_ramp = 0.5 * a_max * t_ramp**2

    if d_total <= 2 * d_ramp:
        t_ramp = np.sqrt(d_total / a_max)
        t_flat = 0
    else:
        t_flat = (d_total - 2 * d_ramp) / v_max

    t_total = 2 * t_ramp + t_flat
    t = np.linspace(0, t_total, n_points)

    position = np.zeros_like(t)
    velocity = np.zeros_like(t)
    acceleration = np.zeros_like(t)

    for i in range(len(t)):
        if t[i] < t_ramp:
            position[i] = 0.5 * a_max * t[i]**2
            velocity[i] = a_max * t[i]
            acceleration[i] = a_max
        elif t[i] < t_ramp + t_flat:
            position[i] = v_max * (t[i] - t_ramp / 2)
            velocity[i] = v_max
            acceleration[i] = 0
        else:
            position[i] = d_total - 0.5 * a_max * (t_total - t[i])**2
            velocity[i] = v_max - a_max * (t[i] - t_ramp - t_flat)
            acceleration[i] = -a_max

    return t, position, velocity, acceleration

In [ ]:
# 예제: 사다리꼴 속도 프로파일
t_trap, pos_trap, vel_trap, acc_trap = trapezoidal_velocity_profile(5.0, 1.0, 0.5)

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
for ax, data, label in zip(axes, [pos_trap, vel_trap, acc_trap],
                           ['Position', 'Velocity', 'Acceleration']):
    ax.plot(t_trap, data)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time [s]')
axes[0].set_title('Trapezoidal Velocity Profile')
plt.tight_layout()
plt.show()

### 2.1 Multi-Joint Trapezoidal Spline

각 조인트에 대해 독립적으로 사다리꼴 프로파일을 적용합니다.

In [ ]:
def trapezoidal_spline(q_init, q_final, max_velocity, max_acceleration, n_points=1000):
    """단일 조인트 사다리꼴 스플라인"""
    delta = q_final - q_init
    t_accel = max_velocity / max_acceleration
    x_accel = 0.5 * max_acceleration * t_accel**2

    if abs(delta) >= 2 * x_accel:
        t_constant = (abs(delta) - 2 * x_accel) / max_velocity
    else:
        t_constant = 0
        t_accel = np.sqrt(abs(delta) / max_acceleration)

    t_total = 2 * t_accel + t_constant
    t = np.linspace(0, t_total, n_points)
    q = np.zeros_like(t)

    mask1 = t < t_accel
    mask2 = (t >= t_accel) & (t < t_accel + t_constant)
    mask3 = t >= t_accel + t_constant

    sign = 1.0 if q_final > q_init else -1.0
    q[mask1] = q_init + sign * 0.5 * max_acceleration * t[mask1]**2
    q[mask2] = q_init + sign * (x_accel + max_velocity * (t[mask2] - t_accel))
    q[mask3] = q_final - sign * 0.5 * max_acceleration * (t_total - t[mask3])**2

    return q, t

# 예제: 6-DOF 조인트 사다리꼴 스플라인
q_init_trap = np.radians([0, -90, 90, 0, 0, 0])
q_final_trap = np.radians([-25, -70, 45, 30, 60, 10])

fig, ax = plt.subplots(figsize=(10, 5))
for j in range(6):
    q, t_j = trapezoidal_spline(q_init_trap[j], q_final_trap[j], 10.0, 10.0)
    ax.plot(np.linspace(0, 1, len(q)), q, label=f'Joint {j+1}')

ax.set_xlabel('Normalized Time')
ax.set_ylabel('Position [rad]')
ax.set_title('Multi-Joint Trapezoidal Spline')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. S-Curve Velocity Profile

S-커브 프로파일은 사다리꼴 프로파일의 가속도 불연속을 해결합니다.
가속도가 연속적으로 변하므로 jerk가 유한하며, 기계적 충격이 줄어듭니다.

In [ ]:
def s_curve_velocity_profile(t_total=1.0, t_ramp=0.3, v_max=1.0, n_per_seg=100):
    """S-커브 속도 프로파일 생성"""
    a_max = v_max / t_ramp

    t1 = np.linspace(0, t_ramp, n_per_seg)
    t2 = np.linspace(t_ramp, t_total - t_ramp, n_per_seg)
    t3 = np.linspace(t_total - t_ramp, t_total, n_per_seg)

    v1 = a_max * t1
    v2 = np.ones_like(t2) * v_max
    v3 = v_max - a_max * (t3 - (t_total - t_ramp))

    t = np.concatenate((t1, t2, t3))
    v = np.concatenate((v1, v2, v3))

    dt = t[1] - t[0]
    p = np.cumsum(v) * dt
    a = np.gradient(v, dt)
    j = np.gradient(a, dt)

    return t, p, v, a, j

t_sc, p_sc, v_sc, a_sc, j_sc = s_curve_velocity_profile()

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(8, 10))
for ax, d, label in zip(axes, [p_sc, v_sc, a_sc, j_sc],
                        ['Position', 'Velocity', 'Acceleration', 'Jerk']):
    ax.plot(t_sc, d)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

axes[0].set_title('S-Curve Velocity Profile')
axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

## 4. Minimum Jerk Trajectory (최소 저크 궤적)

최소 저크 궤적은 jerk의 적분값을 최소화하는 5차 다항식입니다:

$$\theta(t) = \theta_i + (\theta_f - \theta_i)\left(10\left(\frac{t}{T}\right)^3 - 15\left(\frac{t}{T}\right)^4 + 6\left(\frac{t}{T}\right)^5\right)$$

시작/끝에서 속도와 가속도가 0인 quintic polynomial의 특수한 경우입니다.

In [ ]:
def minimum_jerk_trajectory(q_init, q_final, total_time=1.0, dt=0.01):
    """최소 저크 궤적 생성"""
    T = total_time
    t_list, q_list = [], []
    t = 0
    while t < T:
        s = t / T
        q = q_init + (q_final - q_init) * (10*s**3 - 15*s**4 + 6*s**5)
        t_list.append(t)
        q_list.append(q)
        t += dt
    return np.array(t_list), np.array(q_list)

t_jerk, q_jerk = minimum_jerk_trajectory(q_start, q_end, total_time=3.0)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(6):
    ax.plot(t_jerk, q_jerk[:, i], label=f"$q_{i+1}$")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Position [rad]")
ax.set_title("Minimum Jerk Trajectory")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Linear Interpolation (선형 보간)

가장 단순한 궤적 생성 방법. 시작점과 끝점 사이를 직선으로 보간합니다.
속도 불연속이 발생하므로 실제 로봇에서는 주의가 필요합니다.

In [ ]:
def linear_trajectory(start_pos, end_pos, num_points):
    """선형 보간 궤적 생성"""
    n_joints = len(start_pos)
    s = np.linspace(0, 1, num_points)
    trajectory = np.zeros((num_points, n_joints))
    for i in range(n_joints):
        trajectory[:, i] = start_pos[i] + s * (end_pos[i] - start_pos[i])
    return trajectory

traj_linear = linear_trajectory(q_start, q_end, 500)
t_linear = np.linspace(0, 3.0, 500)

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(6):
    ax.plot(t_linear, traj_linear[:, i], label=f"$q_{i+1}$")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Position [rad]")
ax.set_title("Linear Interpolation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Multi-Waypoint Trajectory (다중 경유점 궤적)

여러 경유점(waypoint)을 통과하는 궤적을 생성합니다.
각 구간에 quintic polynomial을 적용하여 연결합니다.

In [ ]:
waypoints = [
    q_start,
    np.radians([23.2, -60.39, 106.5, -46.11, 113.2, 0.0]),
    np.radians([23.2, -71.38,  43.34,  28.04, 113.2, 0.0]),
    np.radians([-20.24, -49.32, 16.11, 33.21, 69.76, 0.0]),
    q_end,
]

seg_duration = 2.0
seg_points = 300
zero6 = np.zeros(6)

all_t, all_pos, all_vel, all_acc = [], [], [], []
t_offset = 0.0

for k in range(len(waypoints) - 1):
    t_seg, pos_seg, vel_seg, acc_seg, _ = quintic_trajectory(
        waypoints[k], zero6, zero6,
        waypoints[k+1], zero6, zero6,
        seg_duration, seg_points
    )
    all_t.append(t_seg + t_offset)
    all_pos.append(pos_seg)
    all_vel.append(vel_seg)
    all_acc.append(acc_seg)
    t_offset += seg_duration

t_all = np.concatenate(all_t)
pos_all = np.concatenate(all_pos)
vel_all = np.concatenate(all_vel)
acc_all = np.concatenate(all_acc)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
titles = ["Joint Positions", "Joint Velocities", "Joint Accelerations"]
data_list = [pos_all, vel_all, acc_all]
ylabels = ["Position [rad]", "Velocity [rad/s]", r"Acceleration [rad/s$^2$]"]

for ax, title, d, ylabel in zip(axes, titles, data_list, ylabels):
    for i in range(6):
        ax.plot(t_all, d[:, i], label=f"$q_{i+1}$")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Multi-Waypoint Quintic Trajectory", fontsize=14)
plt.tight_layout()
plt.show()

## 7. 궤적 생성 방법 비교

Linear, Quintic, Minimum Jerk 세 가지 방법을 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i in range(6):
    axes[0].plot(t_linear, traj_linear[:, i], label=f"$q_{i+1}$")
axes[0].set_title("Linear Interpolation")

for i in range(6):
    axes[1].plot(t, pos[:, i], label=f"$q_{i+1}$")
axes[1].set_title("Quintic Polynomial")

for i in range(6):
    axes[2].plot(t_jerk, q_jerk[:, i], label=f"$q_{i+1}$")
axes[2].set_title("Minimum Jerk")

for ax in axes:
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Position [rad]")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle("Trajectory Generation Methods Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 8. B-Spline Curve

B-스플라인은 제어점(control points)을 통해 부드러운 곡선을 생성합니다.
Task-space 궤적 생성에 유용합니다.

In [ ]:
control_points = np.array([
    (3, 1), (2.5, 4), (0, 1),
    (-2.5, 4), (-3, 0), (-2.5, -4),
    (0, -1), (2.5, -4), (3, -1),
])

x = control_points[:, 0]
y = control_points[:, 1]
n = len(x)
order = 2

t_knot = np.linspace(0, 1, n - (order - 1), endpoint=True)
t_knot = np.append(np.zeros(order), t_knot)
t_knot = np.append(t_knot, np.ones(order))

tck = [t_knot, [x, y], order]
u = np.linspace(0, 1, max(n * 2, 70), endpoint=True)
out = interpolate.splev(u, tck)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y, 'k--', marker='o', markerfacecolor='red', label='Control polygon')
ax.plot(out[0], out[1], 'b', linewidth=2.0, label='B-spline curve')
ax.legend(loc='best')
ax.set_title('B-Spline Curve')
ax.grid(True, alpha=0.3)
ax.axis('equal')
plt.tight_layout()
plt.show()

## Summary

| 방법 | 연속성 | 특징 |
|------|--------|------|
| Linear | $C^0$ | 단순, 속도 불연속 |
| Cubic Polynomial | $C^1$ | 위치/속도 연속 |
| Quintic Polynomial | $C^2$ | 위치/속도/가속도 연속 |
| Trapezoidal | $C^0$ (velocity) | 최대 속도 제한 가능 |
| S-Curve | $C^1$ (velocity) | Jerk 유한, 기계적 충격 감소 |
| Minimum Jerk | $C^2$ | Jerk 최소화, 부드러운 운동 |
| B-Spline | $C^{k-1}$ | 유연한 경로 설계 |